# EcoSort AI - Accuracy Improvement

## 1. Check the environment

Use the isolated local environment. The existing MobileNet remains archived. The new frozen OpenCLIP encoder and trained linear head operate locally, with all 117,893 training images retained.

In [ ]:
from pathlib import Path
import json
import torch
PROJECT=Path.cwd()
assert (PROJECT/'train_semantic.py').exists()
print('MPS available:',torch.backends.mps.is_available())

## 2. Load verified pretrained weights

Download only if absent. Check the pinned revision and SHA-256 before use. Full-object letterboxing preserves bottle ends and other shapes that default center cropping can remove. No image is sent to a remote model.

In [ ]:
%run semantic_model.py

## 3. Cache the unchanged MobileNet predictions

The original class order, manifest order and model hash are preserved for fair comparison and ensemble inference.

In [ ]:
%run cache_mobilenet.py

## 4. Train and select on validation

Extract all training, validation and test features with identical full-object preprocessing. Fit the classification head only on the training partition. Use validation to select regularization and model combinations; reject unconverged heads. Test images never update parameters. Features are resumable and tied to the manifest and preprocessing version.

In [ ]:
%run train_semantic.py

## 5. Inspect the untouched final evaluation

The original ten photos are regression cases. The 55 TACO crops exposed the center-crop defect and are now development diagnostics, not independent evidence. The separate final whole-photo panel was visually reviewed, duplicate-checked and frozen before prediction. Re-running evaluation does not permit replacing failed photos.

In [ ]:
completion=json.loads((PROJECT/'trained_semantic/training_completed.json').read_text())
display(completion)
final=PROJECT/'final_test/evaluation.json'
if final.exists():
    display(json.loads(final.read_text()))
else:
    print('Final external evaluation has not completed.')

## 6. Open the active local application

The inference notebook reads active_model.json and loads the selected model. City of Sydney guidance remains conditional and separate from visual scores. Run only one application instance. An observed test accuracy is not a guarantee for arbitrary future photos.

In [ ]:
import webbrowser
webbrowser.open('http://127.0.0.1:7861')